# Exploration of LAUS Unemployment Data
 
Data documentation notes available online at: https://download.bls.gov/pub/time.series/la/la.txt
Notes from the documentation:
- "Rates and ratios are expressed as percents with one
decimal place."
- location, survey series both in the series_id
    - la.area file contains the location codes
    - la.series file contains the series codes I *think*


Important codes:
- Unemployment rate measure code: 03
- area_type for counties and equivalents: F
- 




In [128]:
# SETUP
import pandas as pd

laus = pd.read_csv(
    "../data_raw/LAUS/la.data.64.County",
    sep="\t"
)

area = pd.read_csv(
    "../data_raw/LAUS/la.area",
    sep="\t"
)

series = pd.read_csv(
    "../data_raw/LAUS/la.series",
    sep="\t"
)

measure = pd.read_csv(
    "../data_raw/LAUS/la.measure",
    sep="\t"
)

area_type = pd.read_csv(
    "../data_raw/LAUS/la.area_type",
    sep="\t"
)


## Main Data Overview + Pre-Clean

### Note:

During my exploratory analysis, I parsed the series_id field directly to better understand the structure of the LAUS identifiers and verify how survey, area, and measure information are encoded. After reviewing the full LAUS documentation in more detail, I realized the la.series and la.area files provide this metadata directly and are meant to be merged with the main data files for this exact purpose. I will be using the offical metadata files in my official cleaning file, but I kept this here to document my exploratory process!

In [129]:
# Checking dataset format
print(laus.head())
print('#################')
print(laus.columns)

# Cleaning up column names
print('#################')
laus.columns = laus.columns.str.strip()


   series_id                       year period         value footnote_codes
0  LAUCN010010000000003            1990    M01           6.5            NaN
1  LAUCN010010000000003            1990    M02           6.4            NaN
2  LAUCN010010000000003            1990    M03           5.6            NaN
3  LAUCN010010000000003            1990    M04           6.6            NaN
4  LAUCN010010000000003            1990    M05           6.1            NaN
#################
Index(['series_id                     ', 'year', 'period', '       value',
       'footnote_codes'],
      dtype='str')
#################


In [ ]:
laus_2 = laus.copy()
# Separating series_id into its respective components according to documentation
laus_2['survey'] = laus_2['series_id'].str[:2]
laus_2['adjusted'] = laus_2['series_id'].str[2:3]
laus_2['area_code'] = laus_2['series_id'].str[3:18]
laus_2['measure'] = laus_2['series_id'].str[18:]

print('#################################')
print(laus_2.head())

#################################
                        series_id  year period         value footnote_codes  \
0  LAUCN010010000000003            1990    M01           6.5            NaN   
1  LAUCN010010000000003            1990    M02           6.4            NaN   
2  LAUCN010010000000003            1990    M03           5.6            NaN   
3  LAUCN010010000000003            1990    M04           6.6            NaN   
4  LAUCN010010000000003            1990    M05           6.1            NaN   

  survey adjusted        area_code       measure  
0     LA        U  CN0100100000000  03            
1     LA        U  CN0100100000000  03            
2     LA        U  CN0100100000000  03            
3     LA        U  CN0100100000000  03            
4     LA        U  CN0100100000000  03            


## Obtaining Area Data Keys to Pull From Main Data
*To use to restrict main dataset to desired area observations*


In [ ]:
# Restricting to county and equivalent codes
area = area[area['area_type_code'] == 'F']

# Separating county and state to use when merging geographic index in
area[['county', 'state']] = area['area_text'].str.split(', ', expand=True)


print(area.head())

     area_type_code        area_code           area_text  display_level  \
1208              F  CN0100100000000  Autauga County, AL              0   
1209              F  CN0100300000000  Baldwin County, AL              0   
1210              F  CN0100500000000  Barbour County, AL              0   
1211              F  CN0100700000000     Bibb County, AL              0   
1212              F  CN0100900000000   Blount County, AL              0   

     selectable  sort_sequence          county state  
1208          T             33  Autauga County    AL  
1209          T             34  Baldwin County    AL  
1210          T             35  Barbour County    AL  
1211          T             36     Bibb County    AL  
1212          T             37   Blount County    AL  


## General Dataset Overview

In [132]:
# Overview
print(laus.head())

                        series_id  year period         value footnote_codes  \
0  LAUCN010010000000003            1990    M01           6.5            NaN   
1  LAUCN010010000000003            1990    M02           6.4            NaN   
2  LAUCN010010000000003            1990    M03           5.6            NaN   
3  LAUCN010010000000003            1990    M04           6.6            NaN   
4  LAUCN010010000000003            1990    M05           6.1            NaN   

  survey adjusted        area_code       measure  
0     LA        U  CN0100100000000  03            
1     LA        U  CN0100100000000  03            
2     LA        U  CN0100100000000  03            
3     LA        U  CN0100100000000  03            
4     LA        U  CN0100100000000  03            


In [133]:
# Stata-esque loop
df = laus.copy()
for col in df.columns:
    print("====", col, "====")
    print("dtype:", df[col].dtype)
    print("nunique:", df[col].nunique())
    print("missing:", df[col].isna().sum())
    print(df[col].value_counts(dropna=False).head(10))
    print("\n")

==== series_id ====
dtype: str
nunique: 12900
missing: 0
series_id
LAUCN060370000000003              471
LAUCN060370000000004              471
LAUCN060370000000005              471
LAUCN060370000000006              471
LAUCN110010000000003              471
LAUCN110010000000004              471
LAUCN110010000000005              471
LAUCN110010000000006              471
LAUCN120860000000003              471
LAUCN120860000000004              471
Name: count, dtype: int64


==== year ====
dtype: int64
nunique: 37
missing: 0
year
2020    167492
2021    167492
2022    167492
2023    167492
2024    167492
2010    167440
2011    167440
2012    167440
2013    167440
2014    167440
Name: count, dtype: int64


==== period ====
dtype: str
nunique: 13
missing: 0
period
M01    476412
M02    476412
M03    463540
M04    463528
M05    463528
M06    463528
M07    463528
M08    463528
M09    463528
M10    463528
Name: count, dtype: int64


==== value ====
dtype: str
nunique: 296532
missing: 0
value
4.0  